## 顶层设计： Chain Class 基类

Chain 是 LangChain 里所有 “流程 / 任务 / 工作流” 的抽象父类。
所有具体的链（LLMChain、SequentialChain、RouterChain...）都必须继承它。
它的作用：统一所有链的行为、统一接口、统一属性、统一调用方式。

类继承关系：

```
Chain --> <name>Chain  # Examples: LLMChain, MapReduceChain, RouterChain
```

```python
# 定义一个名为Chain的基础类
class Chain(Serializable, Runnable[Dict[str, Any], Dict[str, Any]], ABC):
    """为创建结构化的组件调用序列的抽象基类。
    Serializable：能被序列化（存到文件 / 数据库）
    Runnable：能跑！能执行！（支持 invoke 调用、支持管道 | 符号）
    ABC：抽象基类（不能直接 new，必须被继承）

    链应该用来编码对组件的一系列调用，如模型、文档检索器、其他链等，并为此序列提供一个简单的接口。
    把一堆步骤串成一个自动化流程
    
    Chain接口使创建应用程序变得容易，这些应用程序是：
    - 有状态的：给任何Chain添加Memory可以使它具有状态，
    - 可观察的：向Chain传递Callbacks来执行额外的功能，如记录，这在主要的组件调用序列之外。能打印日志、监控、埋点（Callbacks）
    - 可组合的：Chain API足够灵活，可以轻松地将Chains与其他组件结合起来，包括其他Chains。
    
    链公开的主要方法是：
    - `__call__`：链是可以调用的。`__call__`方法是执行Chain的主要方式。它将输入作为一个字典接收，并返回一个字典输出。
    - `run`：一个方便的方法，它以args/kwargs的形式接收输入，并将输出作为字符串或对象返回。这种方法只能用于一部分链，不能像`__call__`那样返回丰富的输出。
    """

    # 调用链
    def invoke(
        self, input: Dict[str, Any], config: Optional[runnableConfig] = None
    ) -> Dict[str, Any]:
        return self(input, **(config or {}))
     """Chain 的标准执行入口
    输入字典 → 执行流程 → 输出字典。
    
    invoke 是 所有 Chain 统一的调用方法，输入输出都是字典，标准化执行入口。
    """
        

    # 链的记忆，保存状态和变量
    memory: Optional[BaseMemory] = None
    """可选的内存对象，默认为None。
    Chain 本身无记忆，加了 memory 就有记忆，实现多轮对话。
    
    Chain 怎么实现记忆？
    答：Chain 基类自带 memory 属性，注入记忆组件后，自动加载历史上下文。
    
    内存是一个在每个链的开始和结束时被调用的类。
    在开始时，内存加载变量并在链中传递它们。在结束时，它保存任何返回的变量。
    有许多不同类型的内存，请查看内存文档以获取完整的目录。"""

    # 回调，可能用于链的某些操作或事件。
    callbacks: Callbacks = Field(default=None, exclude=True)
    """可选的回调处理程序列表（或回调管理器）。默认为None。
    在对链的调用的生命周期中，从on_chain_start开始，到on_chain_end或on_chain_error结束，都会调用回调处理程序。
    每个自定义链可以选择调用额外的回调方法，详细信息请参见Callback文档。
    
    callbacks 用来干嘛？
    答：用于监控链的执行生命周期，做日志、埋点、调试。
    """

    # 是否详细输出模式（调试用）
    verbose: bool = Field(default_factory=_get_verbosity)
    """是否以详细模式运行。在详细模式下，一些中间日志将打印到控制台。默认值为`langchain.verbose`。"""

    # 与链关联的标签
    tags: Optional[List[str]] = None
    """与链关联的可选标签列表，默认为None。
    这些标签将与对这个链的每次调用关联起来，并作为参数传递给在`callbacks`中定义的处理程序。
    你可以使用这些来例如识别链的特定实例与其用例。"""

    # 与链关联的元数据
    metadata: Optional[Dict[str, Any]] = None
    """与链关联的可选元数据，默认为None。
    这些元数据将与对这个链的每次调用关联起来，并作为参数传递给在`callbacks`中定义的处理程序。
    你可以使用这些来例如识别链的特定实例与其用例。"""
```